# Placement feature collision analysis

## 1. 분석 목표와 가설

**이유와 가설.** 기존 16-feature가 같은 selected-vs-unselected pair는 새 모델을
학습하기 전에 표현 중복의 원인을 분리해야 한다. 가설은 두 가지다.

1. 후보가 실제로 다른 점유 cell인데 feature만 같은 경우라면 저장된 geometry가
   collision을 구분할 수 있다.
2. 후보가 같은 점유 cell을 서로 다른 rotation/pivot으로 표현한 경우라면 raw
   coordinate는 구분해도 물리 geometry feature로 채택하면 안 된다.

이 노트북은 저장된 JSONL만 읽고 모델 학습, post-placement simulation과 runtime
통합을 수행하지 않는다.


In [ ]:
EXPERIMENT_NAME = "placement-feature-collision-analysis"
HYPOTHESES = [
    "Stored physical geometry separates genuinely different occupied-cell placements.",
    "Rotation or pivot differences alone reveal representation aliases, not new geometry.",
]
ANALYSIS_USAGE_CONTRACT = {
    "selectedBySource": "label_only",
    "metadata.isWinner": "segmentation_only",
    "winnerPlayerId": "not_used",
    "heuristicBaseline.score": "not_used",
    "metadata.inputSequence": "not_used",
    "foldMetrics": "not_used",
    "futureOutcome": "not_used",
}

# Assertion
assert EXPERIMENT_NAME and len(HYPOTHESES) == 2
assert ANALYSIS_USAGE_CONTRACT["selectedBySource"] == "label_only"
assert ANALYSIS_USAGE_CONTRACT["metadata.isWinner"] == "segmentation_only"

# Output
{"experiment": EXPERIMENT_NAME, "hypotheses": HYPOTHESES}


**해석.** 구분 수치가 높아도 동일 cell 집합의 rotation alias라면 feature 후보로
선정하지 않는다. 이 구분이 전체 분석의 판단 기준이다.


## 2. 환경과 dataset snapshot

**이유와 가설.** baseline과 같은 고정 Python 환경 및 26-game snapshot을 사용해야
collision count 차이를 geometry 효과로 오인하지 않는다.


In [ ]:
import hashlib
import io
import json
import math
import os
import platform
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display

EXPECTED_ENVIRONMENT = {
    "python": "3.12.13",
    "numpy": "2.5.1",
    "pandas": "3.0.3",
    "scipy": "1.18.0",
    "scikitLearn": "1.9.0",
}
ACTUAL_ENVIRONMENT = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikitLearn": sklearn.__version__,
}

def find_repository_root(start):
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "package.json").is_file() and (
            candidate / "environment-ml.yml"
        ).is_file():
            return candidate
    raise AssertionError("REPOSITORY_ROOT_NOT_FOUND")

ROOT = find_repository_root(Path.cwd())
DATASET_RELATIVE_DIR = Path(os.environ.get(
    "PLACEMENT_COLLISION_DATASET",
    "data/datasets/human-behavior-v2-snapshot-26g-20260717",
))
assert not DATASET_RELATIVE_DIR.is_absolute(), "DATASET_PATH_MUST_BE_RELATIVE"
assert ".." not in DATASET_RELATIVE_DIR.parts, "DATASET_PARENT_ALIAS_NOT_ALLOWED"
DATASET_DIR = ROOT / DATASET_RELATIVE_DIR
SOURCE_FILES = {
    "manifest.json": DATASET_DIR / "manifest.json",
    "train.jsonl": DATASET_DIR / "train.jsonl",
    "validation.jsonl": DATASET_DIR / "validation.jsonl",
    "rejected.jsonl": DATASET_DIR / "rejected.jsonl",
}

# Assertion
assert ACTUAL_ENVIRONMENT == EXPECTED_ENVIRONMENT, ACTUAL_ENVIRONMENT
assert DATASET_RELATIVE_DIR.as_posix().endswith("26g-20260717")
assert all(path.is_file() for path in SOURCE_FILES.values())

# Output
display(pd.DataFrame([ACTUAL_ENVIRONMENT]))
display(pd.DataFrame([{"dataset": DATASET_RELATIVE_DIR.as_posix()}]))


**해석.** 분석 환경과 snapshot 이름이 baseline 계약과 다르면 아래 계산을 진행하지
않는다.


## 3. source checksum 검증

**이유와 가설.** 네 source 파일의 SHA-256을 분석 전에 고정해 notebook 실행이
원본을 변경하지 않았음을 마지막 단계에서 byte 단위로 다시 확인한다.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_SHA256_BEFORE = {
    name: sha256_file(path) for name, path in SOURCE_FILES.items()
}
EXPECTED_SOURCE_SHA256 = {
    "manifest.json": "51027be0ebad578f2c38bb8715aa491daa8f13208e3da551a5e4a2dd08bef088",
    "train.jsonl": "85034a66373add54324a56673cb99ed5854810e4299abcaf9c195ae8cbca631e",
    "validation.jsonl": "23b832d95a2057b7b29e7062286c58c68524fa8db00f00870e6076a0a846f58a",
    "rejected.jsonl": "06ede14119014e2b040f8a4d62e00532ca6eecae9424b93e9cc882394db8a85f",
}

# Assertion
assert SOURCE_SHA256_BEFORE == EXPECTED_SOURCE_SHA256, SOURCE_SHA256_BEFORE

# Output
display(pd.DataFrame([
    {"source": name, "sha256": digest}
    for name, digest in SOURCE_SHA256_BEFORE.items()
]))


**해석.** checksum이 다르면 같은 26-game snapshot이 아니므로 baseline count를
억지로 맞추지 않고 중단한다.


## 4. candidate JSONL 직접 로딩

**이유와 가설.** 분석 입력은 exporter나 평가 report가 아니라 snapshot의
`train.jsonl`과 `validation.jsonl` candidate row다. UTF-8/JSON object 계약을 먼저
확인한 뒤 pandas로 직접 읽는다.


In [ ]:
def read_json_object(path):
    value = json.loads(path.read_bytes().decode("utf-8", errors="strict"))
    assert isinstance(value, dict), f"JSON_OBJECT_REQUIRED:{path.name}"
    return value

def read_jsonl_frame(path):
    text = path.read_bytes().decode("utf-8", errors="strict")
    lines = text.splitlines()
    assert lines, f"EMPTY_JSONL:{path.name}"
    for line_number, line in enumerate(lines, start=1):
        assert line.strip(), f"EMPTY_LINE:{path.name}:{line_number}"
        assert isinstance(json.loads(line), dict), (
            f"JSON_OBJECT_LINE_REQUIRED:{path.name}:{line_number}"
        )
    frame = pd.read_json(io.StringIO(text), lines=True, dtype=False)
    assert len(frame) == len(lines), f"PANDAS_ROW_COUNT_MISMATCH:{path.name}"
    return frame

manifest = read_json_object(SOURCE_FILES["manifest.json"])
train_source = read_jsonl_frame(SOURCE_FILES["train.jsonl"])
validation_source = read_jsonl_frame(SOURCE_FILES["validation.jsonl"])
rejected_source = read_jsonl_frame(SOURCE_FILES["rejected.jsonl"])
candidates = pd.concat(
    [train_source.assign(originalSplit="train"),
     validation_source.assign(originalSplit="validation")],
    ignore_index=True,
).copy(deep=True)

# Assertion
assert len(train_source) == manifest["trainSampleCount"]
assert len(validation_source) == manifest["validationSampleCount"]
assert len(rejected_source) == manifest["rejectedPlacementLogCount"]
assert len(candidates) == manifest["sampleCountBySource"]["human"]

# Output
display(pd.DataFrame([
    {"source": "train.jsonl", "rows": len(train_source)},
    {"source": "validation.jsonl", "rows": len(validation_source)},
    {"source": "rejected.jsonl", "rows": len(rejected_source)},
]))


**해석.** 이후 계산은 결합한 candidate row만 사용하며 기존 train/validation split은
수정하거나 새 fold를 만들지 않는다.


## 5. 기존 16개 feature 재생성

**이유와 가설.** collision 기준은 baseline과 같은 이름·순서의 16개 값이다.
`candidateResult`와 `metricDelta`에서 Python으로 직접 재생성하며 label과 metadata를
feature에 포함하지 않는다.


In [ ]:
FEATURE_SCHEMA_VERSION = 1
FEATURE_NAMES = [
    "candidateResult.linesCleared",
    "candidateResult.gameOver",
    "metricDelta.aggregateHeight",
    "metricDelta.maxHeight",
    "metricDelta.holes",
    "metricDelta.bumpiness",
    "metricDelta.columnHeights[0]",
    "metricDelta.columnHeights[1]",
    "metricDelta.columnHeights[2]",
    "metricDelta.columnHeights[3]",
    "metricDelta.columnHeights[4]",
    "metricDelta.columnHeights[5]",
    "metricDelta.columnHeights[6]",
    "metricDelta.columnHeights[7]",
    "metricDelta.columnHeights[8]",
    "metricDelta.columnHeights[9]",
]

def is_integer(value):
    return isinstance(value, (int, np.integer)) and not isinstance(
        value, (bool, np.bool_)
    )

def finite_number(value):
    return (
        isinstance(value, (int, float, np.integer, np.floating))
        and not isinstance(value, (bool, np.bool_))
        and math.isfinite(float(value))
    )

def codepoint_key(value):
    return tuple(ord(character) for character in str(value))

def stable_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def create_decision_key(row):
    assert isinstance(row["gameId"], str) and row["gameId"]
    assert isinstance(row["playerId"], str) and row["playerId"]
    assert is_integer(row["pieceIndex"])
    return f"{row['gameId']}:{row['playerId']}:{int(row['pieceIndex'])}"

def create_placement_key(row):
    placement = row["candidatePlacement"]
    values = [
        placement.get("pieceType"), placement.get("rotation"),
        placement.get("row"), placement.get("col"),
    ]
    assert isinstance(values[0], str) and values[0] == row["pieceType"]
    assert all(is_integer(value) for value in values[1:])
    return f"{values[0]}:{int(values[1])}:{int(values[2])}:{int(values[3])}"

def extract_feature_vector(row):
    result = row["candidateResult"]
    delta = row["metricDelta"]
    assert is_integer(result.get("linesCleared"))
    assert isinstance(result.get("gameOver"), (bool, np.bool_))
    scalar_values = [
        delta.get(name)
        for name in ["aggregateHeight", "maxHeight", "holes", "bumpiness"]
    ]
    columns = delta.get("columnHeights")
    assert all(finite_number(value) for value in scalar_values)
    assert isinstance(columns, list) and len(columns) == 10
    assert all(finite_number(value) for value in columns)
    vector = np.asarray(
        [float(result["linesCleared"]), float(result["gameOver"])]
        + [float(value) for value in scalar_values]
        + [float(value) for value in columns],
        dtype=np.float64,
    )
    assert vector.shape == (16,) and np.isfinite(vector).all()
    return vector

candidates["decisionKey"] = [
    create_decision_key(row) for _, row in candidates.iterrows()
]
candidates["placementKey"] = [
    create_placement_key(row) for _, row in candidates.iterrows()
]
candidates["featureVector"] = [
    extract_feature_vector(row) for _, row in candidates.iterrows()
]
canonical_indices = sorted(
    candidates.index.tolist(),
    key=lambda index: (
        codepoint_key(candidates.at[index, "gameId"]),
        codepoint_key(candidates.at[index, "decisionKey"]),
        codepoint_key(candidates.at[index, "placementKey"]),
    ),
)
candidates = candidates.loc[canonical_indices].reset_index(drop=True)

# Assertion
assert len(FEATURE_NAMES) == len(set(FEATURE_NAMES)) == 16
assert np.isfinite(np.vstack(candidates["featureVector"].to_numpy())).all()

# Output
display(pd.DataFrame({
    "featureIndex": range(len(FEATURE_NAMES)),
    "featureName": FEATURE_NAMES,
}))


**해석.** 이 allowlist 밖의 값은 baseline equality 판정에 관여하지 않는다.
`selectedBySource`는 다음 단계에서 pair label을 식별하는 데만 사용한다.


## 6. decision과 placement 계약 검증

**이유와 가설.** decision마다 정확히 하나의 Human 선택이 있어야 하고, 기존
placementKey는 중복되지 않아야 한다. 동시에 점유 cell 집합은 별도 physical key로
만들어 representation identity와 실제 placement identity를 구분한다.


In [ ]:
REQUIRED_COLUMNS = {
    "datasetSchemaVersion", "gameId", "playerId", "pieceIndex", "source",
    "aiDifficulty", "pieceType", "boardBefore", "candidatePlacement",
    "metricsBefore", "metricDelta", "candidateResult", "selectedBySource",
    "metadata",
}
assert not REQUIRED_COLUMNS.difference(candidates.columns)
assert candidates["datasetSchemaVersion"].map(
    lambda value: is_integer(value) and int(value) == 2
).all()
assert candidates["source"].eq("human").all()
assert candidates["selectedBySource"].map(
    lambda value: isinstance(value, (bool, np.bool_))
).all()

def physical_cell_key(row):
    cells = row["candidatePlacement"].get("cells")
    assert isinstance(cells, list) and len(cells) == 4
    values = []
    for cell in cells:
        assert is_integer(cell.get("row")) and is_integer(cell.get("col"))
        values.append((int(cell["row"]), int(cell["col"])))
    assert len(set(values)) == 4
    return tuple(sorted(values))

def decision_context_signature(row):
    return stable_json({
        "gameId": row["gameId"],
        "playerId": row["playerId"],
        "pieceIndex": int(row["pieceIndex"]),
        "source": row["source"],
        "pieceType": row["pieceType"],
        "aiDifficulty": row["aiDifficulty"],
        "boardBefore": row["boardBefore"],
        "metricsBefore": row["metricsBefore"],
    })

candidates["physicalCellKey"] = [
    physical_cell_key(row) for _, row in candidates.iterrows()
]
candidates["contextSignature"] = [
    decision_context_signature(row) for _, row in candidates.iterrows()
]
decision_contract = candidates.groupby("decisionKey", sort=False).agg(
    candidateCount=("placementKey", "size"),
    selectedCount=("selectedBySource", "sum"),
    placementKeyCount=("placementKey", "nunique"),
    contextCount=("contextSignature", "nunique"),
    gameCount=("gameId", "nunique"),
)

# Assertion
assert (decision_contract["candidateCount"] >= 2).all()
assert (decision_contract["selectedCount"] == 1).all()
assert (decision_contract["placementKeyCount"] == decision_contract["candidateCount"]).all()
assert (decision_contract["contextCount"] == 1).all()
assert (decision_contract["gameCount"] == 1).all()
assert len(decision_contract) == manifest["acceptedPlacementLogCount"]
assert candidates["gameId"].nunique() == manifest["totalGameCount"]

# Output
display(pd.DataFrame([{
    "games": candidates["gameId"].nunique(),
    "decisions": len(decision_contract),
    "candidates": len(candidates),
    "selectedCandidates": int(candidates["selectedBySource"].sum()),
}]))


**해석.** `placementKey` uniqueness는 rotation/pivot 표현의 uniqueness일 뿐이다.
실제 같은 placement인지 여부는 정렬한 네 cell 좌표인 `physicalCellKey`로 판단한다.


## 7. selected-vs-unselected pair 생성

**이유와 가설.** 각 decision의 유일 selected candidate와 모든 unselected candidate를
1:1로 묶는다. 이 단계는 학습 sample이나 weight를 만들지 않는다.


In [ ]:
pair_rows = []
for decision_key in sorted(candidates["decisionKey"].unique(), key=codepoint_key):
    group = candidates.loc[candidates["decisionKey"] == decision_key]
    selected = group.loc[group["selectedBySource"]]
    unselected = group.loc[~group["selectedBySource"]]
    assert len(selected) == 1 and not unselected.empty
    selected_index = int(selected.index[0])
    for unselected_index in unselected.index.tolist():
        pair_rows.append({
            "decisionKey": decision_key,
            "selectedIndex": selected_index,
            "unselectedIndex": int(unselected_index),
        })
pairs = pd.DataFrame(pair_rows)
EXPECTED_BASELINE_SUMMARY = {
    "gameCount": 26,
    "decisionCount": 1158,
    "candidateCount": 39375,
    "selectedCandidateCount": 1158,
    "pairCount": 38217,
    "zeroDifferencePairCount": 1010,
    "zeroDifferenceDecisionCount": 670,
}

# Assertion
assert len(pairs) == len(candidates) - len(decision_contract)
assert len(pairs) == 38_217
assert pairs[["selectedIndex", "unselectedIndex"]].duplicated().sum() == 0

# Output
display(pd.DataFrame([{
    "decisionCount": len(decision_contract),
    "candidateCount": len(candidates),
    "pairCount": len(pairs),
}]))


**해석.** 38,217은 모든 decision의 `(candidateCount - 1)` 합이며 model fold나
sample weight와 무관한 원시 비교 pair 수다.


## 8. baseline zero-difference pair 재현

**이유와 가설.** selected와 unselected의 16-feature vector가 exact equality인 pair만
추출한다. 기준값이 하나라도 다르면 geometry 분석으로 넘어가지 않는다.


In [ ]:
feature_matrix = np.vstack(candidates["featureVector"].to_numpy())
zero_mask = np.asarray([
    np.array_equal(feature_matrix[row.selectedIndex], feature_matrix[row.unselectedIndex])
    for row in pairs.itertuples(index=False)
], dtype=bool)
zero_pairs = pairs.loc[zero_mask].reset_index(drop=True)
zero_pairs["pairId"] = np.arange(len(zero_pairs), dtype=np.int64)
BASELINE_SUMMARY = {
    "gameCount": int(candidates["gameId"].nunique()),
    "decisionCount": int(candidates["decisionKey"].nunique()),
    "candidateCount": int(len(candidates)),
    "selectedCandidateCount": int(candidates["selectedBySource"].sum()),
    "pairCount": int(len(pairs)),
    "zeroDifferencePairCount": int(len(zero_pairs)),
    "zeroDifferenceDecisionCount": int(zero_pairs["decisionKey"].nunique()),
}

# Assertion
assert BASELINE_SUMMARY == EXPECTED_BASELINE_SUMMARY, BASELINE_SUMMARY

# Output
display(pd.DataFrame([BASELINE_SUMMARY]))


**해석.** 1,010 pair와 670 decision이 정확히 재현됐으므로 geometry 원인 분석을
계속할 수 있다.


## 9. geometry 후보 계산

**이유와 가설.** 저장된 placement anchor, rotation, 네 cell 좌표와 `boardBefore`
크기만 사용한다. row는 위에서 아래로 증가한다.

- `row_span = maximum_cell_row - minimum_cell_row`
- `col_span = maximum_cell_col - minimum_cell_col`
- `normalized_placement_row = placement_row / (board_height - 1)`
- `normalized_placement_col = placement_col / (board_width - 1)`
- `normalized_average_cell_height = ((board_height - 1) - mean_cell_row) / (board_height - 1)`
- `left_wall_distance = minimum_cell_col`
- `right_wall_distance = (board_width - 1) - maximum_cell_col`
- `nearest_wall_distance = min(left_wall_distance, right_wall_distance)`


In [ ]:
GEOMETRY_FEATURES = [
    "placement_row", "placement_col", "rotation",
    "minimum_cell_row", "maximum_cell_row", "mean_cell_row",
    "minimum_cell_col", "maximum_cell_col", "mean_cell_col",
    "row_span", "col_span",
    "normalized_placement_row", "normalized_placement_col",
    "normalized_average_cell_height",
    "left_wall_distance", "right_wall_distance", "nearest_wall_distance",
]
INTEGER_GEOMETRY_FEATURES = {
    "placement_row", "placement_col", "rotation",
    "minimum_cell_row", "maximum_cell_row", "minimum_cell_col",
    "maximum_cell_col", "row_span", "col_span",
    "left_wall_distance", "right_wall_distance", "nearest_wall_distance",
}
ANCHOR_ORIENTATION_FEATURES = {
    "placement_row", "placement_col", "rotation",
    "normalized_placement_row", "normalized_placement_col",
}
PHYSICAL_GEOMETRY_FEATURES = set(GEOMETRY_FEATURES).difference(
    ANCHOR_ORIENTATION_FEATURES
)

def calculate_geometry(row):
    board = row["boardBefore"]
    placement = row["candidatePlacement"]
    cells = placement["cells"]
    board_height = len(board)
    board_width = len(board[0])
    assert board_height > 1 and board_width > 1
    assert all(len(board_row) == board_width for board_row in board)
    assert all(is_integer(placement[name]) for name in ["row", "col", "rotation"])
    cell_rows = [int(cell["row"]) for cell in cells]
    cell_cols = [int(cell["col"]) for cell in cells]
    minimum_row, maximum_row = min(cell_rows), max(cell_rows)
    minimum_col, maximum_col = min(cell_cols), max(cell_cols)
    mean_row = float(np.mean(cell_rows))
    mean_col = float(np.mean(cell_cols))
    left_distance = minimum_col
    right_distance = (board_width - 1) - maximum_col
    values = {
        "placement_row": int(placement["row"]),
        "placement_col": int(placement["col"]),
        "rotation": int(placement["rotation"]),
        "minimum_cell_row": minimum_row,
        "maximum_cell_row": maximum_row,
        "mean_cell_row": mean_row,
        "minimum_cell_col": minimum_col,
        "maximum_cell_col": maximum_col,
        "mean_cell_col": mean_col,
        "row_span": maximum_row - minimum_row,
        "col_span": maximum_col - minimum_col,
        "normalized_placement_row": float(placement["row"]) / (board_height - 1),
        "normalized_placement_col": float(placement["col"]) / (board_width - 1),
        "normalized_average_cell_height": (
            (board_height - 1) - mean_row
        ) / (board_height - 1),
        "left_wall_distance": left_distance,
        "right_wall_distance": right_distance,
        "nearest_wall_distance": min(left_distance, right_distance),
    }
    assert list(values) == GEOMETRY_FEATURES
    assert all(math.isfinite(float(value)) for value in values.values())
    assert all(is_integer(values[name]) for name in INTEGER_GEOMETRY_FEATURES)
    assert all(0 <= value < board_height for value in cell_rows)
    assert all(0 <= value < board_width for value in cell_cols)
    return values, board_height, board_width

geometry_records = [calculate_geometry(row) for _, row in candidates.iterrows()]
candidates["geometry"] = [record[0] for record in geometry_records]
candidates["boardHeight"] = [record[1] for record in geometry_records]
candidates["boardWidth"] = [record[2] for record in geometry_records]
for feature in GEOMETRY_FEATURES:
    candidates[f"geometry::{feature}"] = candidates["geometry"].map(
        lambda values: values[feature]
    )

# Assertion
assert set(candidates["boardHeight"]) == {20}
assert set(candidates["boardWidth"]) == {10}
assert all(
    np.isfinite(candidates[f"geometry::{feature}"].to_numpy(dtype=np.float64)).all()
    for feature in GEOMETRY_FEATURES
)

# Output
display(pd.DataFrame([
    {"feature": feature,
     "formulaOrSource": (
         "stored placement value" if feature in {"placement_row", "placement_col", "rotation"}
         else "stored cells and board dimensions"
     )}
    for feature in GEOMETRY_FEATURES
]).head(17))


**해석.** anchor/orientation 계열과 실제 점유 cell에서 파생한 physical geometry를
별도 집합으로 유지한다. 동일 cell을 다른 pivot으로 표현한 차이를 물리 정보로
오인하지 않기 위해서다.


## 10. feature별 collision 분해

**이유와 가설.** 각 geometry 값이 다른 zero-difference pair 수, decision 수, 단독
기여, 다른 geometry와의 중복, 남은 pair 및 전체 candidate 범위를 계산한다.


In [ ]:
for feature in GEOMETRY_FEATURES:
    zero_pairs[f"selected::{feature}"] = zero_pairs["selectedIndex"].map(
        candidates[f"geometry::{feature}"]
    )
    zero_pairs[f"unselected::{feature}"] = zero_pairs["unselectedIndex"].map(
        candidates[f"geometry::{feature}"]
    )
    zero_pairs[f"diff::{feature}"] = (
        zero_pairs[f"selected::{feature}"] != zero_pairs[f"unselected::{feature}"]
    )
diff_columns = [f"diff::{feature}" for feature in GEOMETRY_FEATURES]
different_feature_counts = zero_pairs[diff_columns].sum(axis=1)
FEATURE_ANALYSIS = []
for feature in GEOMETRY_FEATURES:
    mask = zero_pairs[f"diff::{feature}"]
    all_values = candidates[f"geometry::{feature}"].to_numpy(dtype=np.float64)
    FEATURE_ANALYSIS.append({
        "feature": feature,
        "valueType": "integer" if feature in INTEGER_GEOMETRY_FEATURES else "float",
        "finite": bool(np.isfinite(all_values).all()),
        "minimumObservedValue": float(all_values.min()),
        "maximumObservedValue": float(all_values.max()),
        "distinguishedPairCount": int(mask.sum()),
        "distinguishedRatio": float(mask.mean()),
        "affectedDecisionCount": int(zero_pairs.loc[mask, "decisionKey"].nunique()),
        "exclusivePairCount": int((mask & (different_feature_counts == 1)).sum()),
        "overlapPairCount": int((mask & (different_feature_counts > 1)).sum()),
        "remainingPairCount": int((~mask).sum()),
        "distinguishingGameCount": int(zero_pairs.loc[mask, "selectedIndex"].map(
            candidates["gameId"]
        ).nunique()),
    })
FEATURE_ANALYSIS.sort(key=lambda row: (
    -row["distinguishedPairCount"], codepoint_key(row["feature"])
))

# Assertion
assert all(row["finite"] for row in FEATURE_ANALYSIS)
assert next(row for row in FEATURE_ANALYSIS if row["feature"] == "rotation")[
    "distinguishedPairCount"
] == 1_010
assert all(
    next(row for row in FEATURE_ANALYSIS if row["feature"] == feature)[
        "distinguishedPairCount"
    ] == 0
    for feature in PHYSICAL_GEOMETRY_FEATURES
)

# Output
display(pd.DataFrame(FEATURE_ANALYSIS)[[
    "feature", "distinguishedPairCount", "distinguishedRatio",
    "affectedDecisionCount", "exclusivePairCount", "overlapPairCount",
    "remainingPairCount",
]])


**해석.** rotation은 모든 pair를 식별하지만 cell-derived physical geometry는 하나도
식별하지 않는다. 따라서 100% 수치만으로 rotation을 feature로 선택하면 표현 alias를
학습하게 된다.


## 11. feature 간 overlap 분석

**이유와 가설.** 사전 정의한 의미 있는 비교만 사용해 raw coordinate와 cell-derived
정보가 같은 pair를 반복해서 구분하는지 확인한다.


In [ ]:
FEATURE_PAIR_SETS = {
    feature: set(zero_pairs.index[zero_pairs[f"diff::{feature}"]].tolist())
    for feature in GEOMETRY_FEATURES
}
OVERLAP_FAMILIES = {
    "placement_row": {"placement_row"},
    "placement_col": {"placement_col"},
    "rotation": {"rotation"},
    "landing_height_family": {
        "placement_row", "minimum_cell_row", "maximum_cell_row", "mean_cell_row",
        "normalized_placement_row", "normalized_average_cell_height",
    },
    "wall_distance_family": {
        "left_wall_distance", "right_wall_distance", "nearest_wall_distance",
    },
    "cell_span_family": {"row_span", "col_span"},
}

def family_pair_set(name):
    result = set()
    for feature in OVERLAP_FAMILIES[name]:
        result.update(FEATURE_PAIR_SETS[feature])
    return result

OVERLAP_SPECS = [
    ("row_vs_col", "placement_row", "placement_col"),
    ("col_vs_rotation", "placement_col", "rotation"),
    ("landing_height_vs_col", "landing_height_family", "placement_col"),
    ("wall_distance_vs_col", "wall_distance_family", "placement_col"),
    ("cell_span_vs_rotation", "cell_span_family", "rotation"),
]
OVERLAP_ANALYSIS = []
for comparison, left_name, right_name in OVERLAP_SPECS:
    left = family_pair_set(left_name)
    right = family_pair_set(right_name)
    union = left | right
    OVERLAP_ANALYSIS.append({
        "comparison": comparison,
        "left": left_name,
        "right": right_name,
        "leftOnlyPairCount": len(left - right),
        "rightOnlyPairCount": len(right - left),
        "intersectionPairCount": len(left & right),
        "unionPairCount": len(union),
        "remainingPairCount": len(zero_pairs) - len(union),
    })

# Assertion
assert len(OVERLAP_ANALYSIS) == len(OVERLAP_SPECS) == 5
assert next(row for row in OVERLAP_ANALYSIS if row["comparison"] == "col_vs_rotation")[
    "unionPairCount"
] == 1_010

# Output
display(pd.DataFrame(OVERLAP_ANALYSIS))


**해석.** col 차이는 rotation 차이의 부분집합이고 wall distance와 cell span은 추가
정보를 주지 않는다. 조합은 model feature bundle 제안이 아니라 정보 중복 진단이다.


## 12. pieceType/game/difficulty/outcome segmentation

**이유와 가설.** collision이 piece, rotation representation, 난이도, outcome, game과
candidate 수에 집중되는지 denominator와 함께 본다. outcome은 evaluation
segmentation일 뿐 geometry 계산에 사용하지 않는다.


In [ ]:
selected_candidates = candidates.loc[candidates["selectedBySource"]].copy()
candidate_counts = candidates.groupby("decisionKey", sort=False).size()
decision_summary = selected_candidates[[
    "decisionKey", "gameId", "pieceType", "aiDifficulty", "metadata"
]].copy()
decision_summary["candidateCount"] = decision_summary["decisionKey"].map(candidate_counts)
decision_summary["outcome"] = decision_summary["metadata"].map(
    lambda value: "winner" if value["isWinner"] else "loss"
)
decision_summary["selectedRotation"] = selected_candidates["geometry::rotation"].to_numpy()
decision_summary["candidateCountBin"] = decision_summary["candidateCount"].map(
    lambda value: (
        "under_20" if value < 20 else
        "20_to_29" if value < 30 else
        "30_to_39" if value < 40 else
        "40_or_more"
    )
)
for column in ["gameId", "pieceType", "aiDifficulty", "outcome", "candidateCountBin"]:
    zero_pairs[column] = zero_pairs["selectedIndex"].map(candidates[column]) if (
        column in candidates.columns
    ) else zero_pairs["decisionKey"].map(
        decision_summary.set_index("decisionKey")[column]
    )
zero_pairs["selectedRotation"] = zero_pairs["selectedIndex"].map(
    candidates["geometry::rotation"]
)
zero_pairs["unselectedRotation"] = zero_pairs["unselectedIndex"].map(
    candidates["geometry::rotation"]
)
zero_pairs["rotationPair"] = zero_pairs.apply(
    lambda row: f"{int(row['selectedRotation'])}->{int(row['unselectedRotation'])}",
    axis=1,
)

def summarize_segment(segment_name, decision_column, pair_column=None):
    if pair_column is None:
        pair_column = decision_column
    values = set(decision_summary[decision_column].tolist()) | set(
        zero_pairs[pair_column].tolist()
    )
    rows = []
    collision_decisions = set(zero_pairs["decisionKey"])
    for value in sorted(values, key=codepoint_key):
        decisions = decision_summary.loc[decision_summary[decision_column] == value]
        pair_subset = zero_pairs.loc[zero_pairs[pair_column] == value]
        collision_count = int(decisions["decisionKey"].isin(collision_decisions).sum())
        rows.append({
            "segment": segment_name,
            "value": str(value),
            "totalDecisionCount": int(len(decisions)),
            "collisionDecisionCount": collision_count,
            "collisionDecisionRate": float(collision_count / len(decisions)) if len(decisions) else 0.0,
            "zeroDifferencePairCount": int(len(pair_subset)),
            "zeroDifferencePairShare": float(len(pair_subset) / len(zero_pairs)),
            "gameCount": int(decisions["gameId"].nunique()),
        })
    return sorted(rows, key=lambda row: (-row["zeroDifferencePairCount"], codepoint_key(row["value"])))

SEGMENTATION = {
    "pieceType": summarize_segment("pieceType", "pieceType"),
    "selectedRotation": summarize_segment("selectedRotation", "selectedRotation"),
    "aiDifficulty": summarize_segment("aiDifficulty", "aiDifficulty"),
    "outcome": summarize_segment("outcome", "outcome"),
    "gameId": summarize_segment("gameId", "gameId"),
    "candidateCountBin": summarize_segment("candidateCountBin", "candidateCountBin"),
}
rotation_pair_counts = (
    zero_pairs.groupby("rotationPair", sort=False)
    .agg(zeroDifferencePairCount=("pairId", "size"),
         collisionDecisionCount=("decisionKey", "nunique"),
         gameCount=("gameId", "nunique"))
    .reset_index()
)
rotation_pair_counts["zeroDifferencePairShare"] = (
    rotation_pair_counts["zeroDifferencePairCount"] / len(zero_pairs)
)
rotation_pair_counts = rotation_pair_counts.sort_values(
    ["zeroDifferencePairCount", "rotationPair"], ascending=[False, True]
)
WINNER_GAME_COUNT = int(decision_summary.loc[
    decision_summary["outcome"] == "winner", "gameId"
].nunique())
top_game = SEGMENTATION["gameId"][0]

# Assertion
assert WINNER_GAME_COUNT == 3
assert sum(row["zeroDifferencePairCount"] for row in SEGMENTATION["pieceType"]) == 1_010
assert top_game["zeroDifferencePairCount"] == 76

# Output
display(pd.DataFrame(SEGMENTATION["pieceType"]))
display(pd.DataFrame(SEGMENTATION["aiDifficulty"]))
display(pd.DataFrame(SEGMENTATION["outcome"]))
display(pd.DataFrame(SEGMENTATION["candidateCountBin"]))
display(pd.DataFrame(SEGMENTATION["gameId"]).head(8))
display(rotation_pair_counts.head(10))


**해석.** collision은 rotational symmetry가 있는 square/ess/zee/bar에만 나타난다.
가장 큰 game도 전체 pair의 약 7.5%이며 한 game만의 현상은 아니다. winner는 3게임뿐이므로
winner/loss 차이는 일반화하지 않는다.


## 13. geometry로 해결되지 않는 pair 분석

**이유와 가설.** raw rotation/pivot이 다른 것과 실제 점유 cell geometry가 다른 것을
분리한다. physical geometry가 모두 같으면 모델 feature가 아니라 candidate 표현
중복 문제다.


In [ ]:
zero_pairs["samePlacementKey"] = zero_pairs.apply(
    lambda row: candidates.at[row["selectedIndex"], "placementKey"]
    == candidates.at[row["unselectedIndex"], "placementKey"],
    axis=1,
)
zero_pairs["samePhysicalCellSet"] = zero_pairs.apply(
    lambda row: candidates.at[row["selectedIndex"], "physicalCellKey"]
    == candidates.at[row["unselectedIndex"], "physicalCellKey"],
    axis=1,
)
anchor_union = set().union(*[
    FEATURE_PAIR_SETS[feature] for feature in ANCHOR_ORIENTATION_FEATURES
])
physical_union = set().union(*[
    FEATURE_PAIR_SETS[feature] for feature in PHYSICAL_GEOMETRY_FEATURES
])
UNRESOLVED_ANALYSIS = {
    "samePlacementKeyPairCount": int(zero_pairs["samePlacementKey"].sum()),
    "samePhysicalCellSetPairCount": int(zero_pairs["samePhysicalCellSet"].sum()),
    "differentPlacementRowPairCount": int(zero_pairs["diff::placement_row"].sum()),
    "differentPlacementColPairCount": int(zero_pairs["diff::placement_col"].sum()),
    "differentRotationPairCount": int(zero_pairs["diff::rotation"].sum()),
    "anchorOrientationResolvedPairCount": int(len(anchor_union)),
    "anchorOrientationUnresolvedPairCount": int(len(zero_pairs) - len(anchor_union)),
    "physicalGeometryResolvedPairCount": int(len(physical_union)),
    "physicalGeometryUnresolvedPairCount": int(len(zero_pairs) - len(physical_union)),
}

# Assertion
assert UNRESOLVED_ANALYSIS == {
    "samePlacementKeyPairCount": 0,
    "samePhysicalCellSetPairCount": 1010,
    "differentPlacementRowPairCount": 90,
    "differentPlacementColPairCount": 410,
    "differentRotationPairCount": 1010,
    "anchorOrientationResolvedPairCount": 1010,
    "anchorOrientationUnresolvedPairCount": 0,
    "physicalGeometryResolvedPairCount": 0,
    "physicalGeometryUnresolvedPairCount": 1010,
}

# Output
display(pd.DataFrame([UNRESOLVED_ANALYSIS]))


**해석.** 기존 placementKey로는 1,010개가 모두 다르지만 실제 네 cell은 모두 같다.
rotation과 pivot 좌표는 representation alias를 식별할 뿐이며 physical geometry로는
1,010개가 전부 unresolved다.


## 14. 단일 feature 후보 선정

**이유와 가설.** raw 구분률 외에 game 일관성, inference 가능성, simulation 의존성,
설명 가능성, 절대 위치 편향, 정보 중복과 scale을 함께 평가한다. 동일 physical
placement의 ID를 맞히는 값은 geometry feature 후보가 아니다.


In [ ]:
SELECTION_CRITERIA = [
    "collision_pair_separation",
    "cross_game_consistency",
    "runtime_computability",
    "no_python_simulation",
    "semantic_explainability",
    "absolute_position_bias_risk",
    "information_overlap",
    "stable_scale_and_range",
]
SELECTION_STATUS = "geometry_does_not_resolve_collision"
SINGLE_FEATURE_SELECTION = {
    "status": SELECTION_STATUS,
    "selectedFeature": None,
    "highestRawSeparator": "rotation",
    "highestRawSeparatorPairCount": 1010,
    "physicalGeometrySeparatorCount": 0,
    "reason": (
        "All zero-difference pairs occupy identical cells. Rotation and anchor coordinates "
        "only distinguish symmetric representation aliases, so adding them would not add "
        "physical placement information."
    ),
    "nextGate": (
        "Canonicalize or evaluate identical occupied-cell candidates as one equivalence "
        "class before any feature/model comparison."
    ),
}

# Assertion
assert len(SELECTION_CRITERIA) == 8
assert SINGLE_FEATURE_SELECTION["selectedFeature"] is None
assert SELECTION_STATUS == "geometry_does_not_resolve_collision"
assert UNRESOLVED_ANALYSIS["physicalGeometryUnresolvedPairCount"] == 1_010

# Output
display(pd.DataFrame([SINGLE_FEATURE_SELECTION]))


**해석.** rotation은 계산 가능하고 scale도 작지만 의미 계약을 통과하지 못한다.
이번 실험의 결론은 새 feature 추가가 아니라 physical-cell equivalence를 먼저 다루는
것이다.


## 15. 결정성 및 원본 불변성

**이유와 가설.** source checksum을 다시 확인하고 timestamp·사용자 절대 경로가 없는
결정적 JSON report를 저장소 밖에 atomic write한다. 같은 입력으로 두 번 실행하면
report가 byte-identical이어야 한다.


In [ ]:
SOURCE_SHA256_AFTER = {
    name: sha256_file(path) for name, path in SOURCE_FILES.items()
}
REPORT_PATH = Path(os.environ.get(
    "PLACEMENT_COLLISION_REPORT",
    "/private/tmp/placement-feature-collision-analysis-report.json",
))
if not REPORT_PATH.is_absolute():
    REPORT_PATH = ROOT / REPORT_PATH

def lexical_path(path):
    return Path(os.path.abspath(os.fspath(path)))

def same_inode(left, right):
    left_stat = left.stat()
    right_stat = right.stat()
    return (left_stat.st_dev, left_stat.st_ino) == (right_stat.st_dev, right_stat.st_ino)

def assert_report_path_safe(output_path, source_paths):
    output_lexical = lexical_path(output_path)
    output_real = output_path.resolve(strict=False)
    assert not output_real.is_relative_to(ROOT.resolve()), "REPORT_MUST_BE_OUTSIDE_REPOSITORY"
    if os.path.lexists(output_path):
        assert not output_path.is_symlink(), "REPORT_SYMLINK_NOT_ALLOWED"
    for source_path in source_paths:
        assert output_lexical != lexical_path(source_path), "REPORT_LEXICAL_SOURCE_COLLISION"
        assert output_real != source_path.resolve(strict=True), "REPORT_REALPATH_SOURCE_COLLISION"
        if os.path.lexists(output_path):
            assert not same_inode(output_path, source_path), "REPORT_INODE_SOURCE_COLLISION"

def atomic_write_report(output_path, text, source_paths):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    assert_report_path_safe(output_path, source_paths)
    descriptor, staging_name = tempfile.mkstemp(
        prefix=f".{output_path.name}.", suffix=".tmp", dir=output_path.parent
    )
    staging_path = Path(staging_name)
    descriptor_identity = os.fstat(descriptor)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline=chr(10)) as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        staging_identity = staging_path.lstat()
        assert not staging_path.is_symlink(), "REPORT_STAGING_SYMLINK_NOT_ALLOWED"
        assert (
            staging_identity.st_dev, staging_identity.st_ino
        ) == (
            descriptor_identity.st_dev, descriptor_identity.st_ino
        ), "REPORT_STAGING_IDENTITY_CHANGED"
        assert all(not same_inode(staging_path, source) for source in source_paths)
        assert_report_path_safe(output_path, source_paths)
        os.replace(staging_path, output_path)
    finally:
        if staging_path.exists():
            staging_path.unlink()

report = {
    "analysisReportSchemaVersion": 1,
    "experimentName": EXPERIMENT_NAME,
    "analysisStatus": SELECTION_STATUS,
    "modelTrainingPerformed": False,
    "runtimeIntegrationPerformed": False,
    "sourceDataset": {
        "relativeDirectory": DATASET_RELATIVE_DIR.as_posix(),
        "sourceFileSha256": SOURCE_SHA256_BEFORE,
        "sourceUnchanged": SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE,
    },
    "environment": ACTUAL_ENVIRONMENT,
    "analysisUsageContract": ANALYSIS_USAGE_CONTRACT,
    "baseline": BASELINE_SUMMARY,
    "featureContract": {
        "featureSchemaVersion": FEATURE_SCHEMA_VERSION,
        "featureNames": FEATURE_NAMES,
    },
    "geometryContract": {
        "geometryFeatures": GEOMETRY_FEATURES,
        "anchorOrientationFeatures": sorted(ANCHOR_ORIENTATION_FEATURES, key=codepoint_key),
        "physicalGeometryFeatures": sorted(PHYSICAL_GEOMETRY_FEATURES, key=codepoint_key),
        "boardRowDirection": "top_to_bottom_increasing",
        "normalization": {
            "normalizedPlacementRow": "placement_row/(board_height-1)",
            "normalizedPlacementCol": "placement_col/(board_width-1)",
            "normalizedAverageCellHeight": "((board_height-1)-mean_cell_row)/(board_height-1)",
        },
    },
    "placementIdentity": UNRESOLVED_ANALYSIS,
    "featureAnalysis": FEATURE_ANALYSIS,
    "overlapAnalysis": OVERLAP_ANALYSIS,
    "segmentation": {
        **SEGMENTATION,
        "rotationPair": rotation_pair_counts.to_dict(orient="records"),
        "winnerGameCount": WINNER_GAME_COUNT,
        "largestGameCollisionPairShare": top_game["zeroDifferencePairShare"],
    },
    "selectionCriteria": SELECTION_CRITERIA,
    "singleFeatureSelection": SINGLE_FEATURE_SELECTION,
    "nextExperimentScope": {
        "first": "Define occupied-cell equivalence classes per decision.",
        "comparison": (
            "Recompute representation diagnostics and held-out ranking with identical "
            "occupied-cell candidates grouped, keeping the same 16 features, dataset, "
            "LOGO folds, scaler, estimator, and metrics."
        ),
        "excluded": [
            "new_geometry_feature", "new_model", "production_artifact", "runtime_change"
        ],
    },
}
report_text = json.dumps(
    report, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False
) + chr(10)

# Assertion
assert SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE, "SOURCE_DATASET_MUTATED"
assert "timestamp" not in report_text.lower()
assert str(ROOT) not in report_text
assert_report_path_safe(REPORT_PATH, list(SOURCE_FILES.values()))
atomic_write_report(REPORT_PATH, report_text, list(SOURCE_FILES.values()))
REPORT_SHA256 = sha256_file(REPORT_PATH)
assert REPORT_SHA256 == hashlib.sha256(report_text.encode("utf-8")).hexdigest()

# Output
display(pd.DataFrame([{
    "reportSha256": REPORT_SHA256,
    "sourceUnchanged": SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE,
    "analysisStatus": SELECTION_STATUS,
}]))


**해석.** 실행별로 달라지는 값은 report에 포함하지 않는다. 두 headless run의 report
SHA-256과 source 전후 checksum 비교가 최종 결정성 증거다.


## 16. 결론과 다음 실험 게이트

**이유와 가설.** feature collision의 원인이 실제 geometry 부족인지 candidate
representation 중복인지에 따라 다음 실험 범위가 달라진다.


In [ ]:
FINAL_GATE = {
    "status": SELECTION_STATUS,
    "baselineReproduced": BASELINE_SUMMARY == EXPECTED_BASELINE_SUMMARY,
    "samePhysicalCellSetPairCount": UNRESOLVED_ANALYSIS["samePhysicalCellSetPairCount"],
    "physicalGeometryUnresolvedPairCount": UNRESOLVED_ANALYSIS[
        "physicalGeometryUnresolvedPairCount"
    ],
    "selectedGeometryFeature": SINGLE_FEATURE_SELECTION["selectedFeature"],
    "modelTrainingAllowedInThisTask": False,
    "nextGate": SINGLE_FEATURE_SELECTION["nextGate"],
}

# Assertion
assert FINAL_GATE["baselineReproduced"] is True
assert FINAL_GATE["samePhysicalCellSetPairCount"] == 1_010
assert FINAL_GATE["physicalGeometryUnresolvedPairCount"] == 1_010
assert FINAL_GATE["selectedGeometryFeature"] is None
assert FINAL_GATE["modelTrainingAllowedInThisTask"] is False

# Output
display(pd.DataFrame([FINAL_GATE]))


**해석.** 상태는 `geometry_does_not_resolve_collision`이다. 1,010 pair는 서로 다른
실제 placement가 아니라 대칭 조각의 동일 occupied-cell placement를 다른 rotation과
pivot으로 표현한 결과다. 다음 gate는 feature 추가가 아니라 occupied-cell equivalence
정의와 동일 baseline 조건의 representation 비교다.
